In [ ]:
import pandas as pd
import scipy.stats as stats
from dotenv import load_dotenv
import mysql.connector
import os


In [ ]:
load_dotenv()

connection = mysql.connector.connect(
    host=os.getenv("DB_HOST"),
    user=os.getenv("DB_USER"),
    password=os.getenv("DB_PASSWORD"),
    database=os.getenv("DB_NAME")
)

cursor = connection.cursor()

In [ ]:

query = """
SELECT
    o.order_id,
    o.order_delivered_customer_date,
    o.order_estimated_delivery_date,
    r.review_score
FROM orders o
JOIN order_reviews r
    ON o.order_id = r.order_id
WHERE o.order_delivered_customer_date IS NOT NULL
  AND o.order_estimated_delivery_date IS NOT NULL
  AND r.review_score IS NOT NULL;
"""

df_stats = pd.read_sql(query, connection)
df_stats.head()

In [ ]:
df_stats["delivery_delay_days"] = (
    pd.to_datetime(df_stats["order_delivered_customer_date"])
    - pd.to_datetime(df_stats["order_estimated_delivery_date"])
).dt.days

df_stats["delivery_status"] = df_stats["delivery_delay_days"].apply(
    lambda days: "Delayed" if days > 0 else "On-Time"
)

df_stats.groupby("delivery_status")["review_score"].agg(
    ["count", "mean", "std"]
)

In [ ]:
delayed_scores = df_stats.loc[
    df_stats['delivery_status'] == "Delayed" , "review_score"
]

on_time_scores = df_stats.loc[
    df_stats['delivery_status'] == "On-Time" ,"review_score"
]

t_stats ,p_value = stats.ttest_ind(
    delayed_scores,
    on_time_scores,
    equal_var=False
)

print("Deleyed Orders Mean score: " , delayed_scores.mean())
print("On Time Order Mean score: " , on_time_scores.mean())
print("T-statistic: " , t_stats)
print('p_value: ' , p_value)


if p_value < 0.05 :
        print("Result: The difference is statistically significant.")
else:
    print("Result: The difference is not statistically significant.")

In [ ]:
'''
Interpretation
- Delayed orders: Average review score = 2.27 / 5
- On-time orders: Average review score = 4.29 / 5
- P-value: Approximately 0 (extremely small, below 0.05)
- Conclusion: The difference is statistically significant. 
  Delayed orders are associated with substantially lower review scores.
'''

In [ ]:
# ANOVA — Product Category & Spending
query = """
SELECT
    COALESCE(
        pct.product_category_name_english,
        p.product_category_name,
        'Unknown'
    ) AS product_category,
    oi.price + oi.freight_value AS order_value
FROM order_items oi
JOIN products p
    ON oi.product_id = p.product_id
LEFT JOIN product_category_translation pct
    ON p.product_category_name = pct.product_category_name;
"""

df_anova = pd.read_sql(query, connection)

df_anova.head()

In [9]:
category_groups =[
    group["order_value"].values
    for _, group in df_anova.groupby('product_category')
    if len(group) >=2
]

f_stat,p_value = stats.f_oneway(*category_groups)

print("F-statistics: " ,f_stat)
print("P-value :" ,p_value)

if p_value < 0.05:
    print("Result: Average order value differs significantly across product categories.")
else:
    print("Result: No statistically significant difference was found.")


F-statistics:  192.39098224264768
P-value : 0.0
Result: Average order value differs significantly across product categories.


In [ ]:
'''
- F-statistic: 192.39
- P-value: Approximately 0, below 0.05
- Conclusion: Average order-item value differs significantly across product categories.

The One-Way ANOVA test indicates statistically significant differences in average order-item value 
across product categories. This suggests that spending varies by category, 
although ANOVA alone does not identify which specific categories differ.
'''

In [10]:
# Chi-Square — Payment Method & Order Status
query = """
SELECT
    op.payment_type,
    o.order_status,
    COUNT(DISTINCT o.order_id) AS order_count
FROM orders o
JOIN order_payment op
    ON o.order_id = op.order_id
GROUP BY
    op.payment_type,
    o.order_status;
"""

df_chi = pd.read_sql(query, connection)

df_chi.head()

C:\Users\iseak\AppData\Local\Temp\ipykernel_8012\1367875688.py:15: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_chi = pd.read_sql(query, connection)


,payment_type,order_status,order_count
0,boleto,canceled,95
1,boleto,created,2
2,boleto,delivered,19191
3,boleto,invoiced,67
4,boleto,processing,70


In [11]:
contingency_table = df_chi.pivot_table(
    index="payment_type",
    columns="order_status",
    values="order_count",
    aggfunc="sum",
    fill_value=0
)

chi2_stat ,p_value,dof,expected = stats.chi2_contingency(
    contingency_table
)

print("Chi-Square Statistic:", chi2_stat)
print("Degrees of Freedom:", dof)
print("P-value:", p_value)

if p_value < 0.05:
    print("Result: Payment method and order status are significantly associated.")
else:
    print("Result: No statistically significant association was found.")


Chi-Square Statistic: 700.4612971025354
Degrees of Freedom: 28
P-value: 1.568423441380726e-129
Result: Payment method and order status are significantly associated.


In [ ]:
'''
Chi-Square Statistic: 700.46
Degrees of Freedom: 28
P-value: \(1.57 \times 10^{-129}\)
Conclusion: There is a statistically significant association between payment method and order status.
'''